<a href="https://colab.research.google.com/github/ricomelgozajjesus/monografia-armonica/blob/main/python-lab/notebooks/Notebook_VII_02_reconstruccion_P_floquet.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Notebook VII.2 — Reconstrucción numérica de la parte periódica $P(t)$ en la teoría de Floquet

Este notebook acompaña la **Ventana Computacional VII.2** de la monografía.

La idea central es construir numéricamente la descomposición de Floquet

$$
\Phi(t)=P(t)e^{Rt},
$$

para un sistema lineal periódico, y reconstruir explícitamente

$$
P(t)=\Phi(t)e^{-Rt}.
$$

El objetivo es ver que $P(t)$ no es una abstracción: se puede calcular, graficar y verificar.

## 1. Sistema de prueba: oscilador con rigidez periódica

Usaremos el oscilador lineal paramétrico

$$
\ddot q + \bigl(\omega_0^2+\epsilon\cos(\Omega t)\bigr)q=0.
$$

En variables de estado,

$$
x(t)=\begin{bmatrix}q(t)\\ \dot q(t)\end{bmatrix},
$$

se obtiene

$$
\dot x=A(t)x,
\qquad
A(t)=\begin{bmatrix}
0 & 1\\
-\omega_0^2-\epsilon\cos(\Omega t) & 0
\end{bmatrix}.
$$

La matriz $A(t)$ es periódica con período

$$
T=\frac{2\pi}{\Omega}.
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.linalg import logm, expm, eig

# ------------------------------------------------------------
# Parámetros del oscilador periódico
# ------------------------------------------------------------
omega0 = 1.0
Omega = 1.0
epsilon = 0.35

T = 2*np.pi/Omega
n_periods = 4

print(f"omega0  = {omega0}")
print(f"Omega   = {Omega}")
print(f"epsilon = {epsilon}")
print(f"T       = {T:.8f}")

In [ ]:
def A(t):
    """Matriz periódica del sistema."""
    return np.array([
        [0.0, 1.0],
        [-(omega0**2 + epsilon*np.cos(Omega*t)), 0.0]
    ])

# Visualización de la entrada periódica de A(t)
t_plot = np.linspace(0, n_periods*T, 1200)
a21 = np.array([A(tt)[1,0] for tt in t_plot])

plt.figure(figsize=(9, 3.5))
plt.plot(t_plot, a21)
for k in range(n_periods + 1):
    plt.axvline(k*T, linestyle=":", linewidth=1)
plt.xlabel("t")
plt.ylabel(r"$A_{21}(t)$")
plt.title(r"Coeficiente periódico $A_{21}(t)=-(\omega_0^2+\epsilon\cos(\Omega t))$")
plt.grid(True)
plt.show()

## 2. Integración de la matriz fundamental

La matriz fundamental satisface

$$
\dot \Phi(t)=A(t)\Phi(t),
\qquad
\Phi(0)=I.
$$

En dimensión dos, se integra como un sistema de cuatro ecuaciones escalares. Numéricamente, representaremos $\Phi(t)$ como un vector de cuatro entradas y lo reacomodaremos como matriz cuando sea necesario.

In [ ]:
def phi_rhs(t, y):
    Phi = y.reshape((2, 2))
    dPhi = A(t) @ Phi
    return dPhi.reshape(4)

Phi0 = np.eye(2).reshape(4)

t_span = (0.0, n_periods*T)
t_eval = np.linspace(t_span[0], t_span[1], 4001)

sol_phi = solve_ivp(
    phi_rhs,
    t_span,
    Phi0,
    t_eval=t_eval,
    rtol=1e-10,
    atol=1e-12
)

if not sol_phi.success:
    raise RuntimeError(sol_phi.message)

t = sol_phi.t
Phi_t = sol_phi.y.T.reshape((-1, 2, 2))

print("Número de muestras:", len(t))
print("Phi(0) =")
print(Phi_t[0])

## 3. Matriz de monodromía y matriz efectiva $R$

La matriz de monodromía es

$$
M=\Phi(T).
$$

A partir de ella se construye una matriz efectiva

$$
R=\frac{1}{T}\log(M),
$$

que satisface, para la rama elegida del logaritmo matricial,

$$
e^{RT}=M.
$$

Esta matriz $R$ representa la dinámica efectiva vista de período en período.

In [ ]:
# Integramos exactamente hasta T para obtener una monodromía precisa
sol_T = solve_ivp(
    phi_rhs,
    (0.0, T),
    Phi0,
    t_eval=[T],
    rtol=1e-11,
    atol=1e-13
)

M = sol_T.y[:, -1].reshape((2, 2))

print("M = Phi(T) =")
print(M)
print("\ndet(M) =", np.linalg.det(M))

multipliers, eigvecs = eig(M)
print("\nMultiplicadores de Floquet:")
for i, mu in enumerate(multipliers, start=1):
    print(f"mu_{i} = {mu:.12g}, |mu_{i}| = {abs(mu):.12g}")

R = logm(M)/T
R = np.real_if_close(R, tol=1000)

print("\nR = log(M)/T =")
print(R)

err_exp = np.linalg.norm(expm(R*T) - M)
print("\n||exp(RT)-M|| =", err_exp)

## 4. Reconstrucción de $P(t)$

La descomposición de Floquet dice que

$$
\Phi(t)=P(t)e^{Rt}.
$$

Por lo tanto,

$$
P(t)=\Phi(t)e^{-Rt}.
$$

Esta es la operación central de este notebook.

In [ ]:
P_t = np.array([
    Phi_t[i] @ expm(-R*t[i])
    for i in range(len(t))
])
P_t = np.real_if_close(P_t, tol=1000)

print("P(0) =")
print(P_t[0])

# Verificación de reconstrucción: Phi(t) ?= P(t) exp(Rt)
reconstruction_error = np.array([
    np.linalg.norm(P_t[i] @ expm(R*t[i]) - Phi_t[i])
    for i in range(len(t))
])

print("\nMáximo error de reconstrucción ||Phi - P exp(Rt)||:")
print(np.max(reconstruction_error))

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(t, P_t[:, 0, 0], label=r"$P_{11}(t)$")
plt.plot(t, P_t[:, 0, 1], label=r"$P_{12}(t)$")
plt.plot(t, P_t[:, 1, 0], label=r"$P_{21}(t)$")
plt.plot(t, P_t[:, 1, 1], label=r"$P_{22}(t)$")

for k in range(n_periods + 1):
    plt.axvline(k*T, linestyle=":", linewidth=1)

plt.xlabel("t")
plt.ylabel("entradas de P(t)")
plt.title(r"Parte periódica reconstruida $P(t)=\Phi(t)e^{-Rt}$")
plt.legend(ncol=2)
plt.grid(True)
plt.show()

## 5. Verificación de periodicidad de $P(t)$

Si la reconstrucción es correcta, debe cumplirse

$$
P(t+T)=P(t).
$$

Como estamos trabajando sobre una malla uniforme con un número entero de períodos, podemos comparar muestras separadas por un período.

In [ ]:
# Como t_eval fue construido con 4001 muestras en 4 períodos,
# hay 1000 intervalos por período.
samples_per_period = (len(t) - 1) // n_periods
print("Muestras por período:", samples_per_period)

periodicity_errors = []
t_periodicity = []

for i in range(len(t) - samples_per_period):
    periodicity_errors.append(np.linalg.norm(P_t[i + samples_per_period] - P_t[i]))
    t_periodicity.append(t[i])

periodicity_errors = np.array(periodicity_errors)
t_periodicity = np.array(t_periodicity)

plt.figure(figsize=(9, 3.5))
plt.plot(t_periodicity, periodicity_errors)
plt.xlabel("t")
plt.ylabel(r"$\|P(t+T)-P(t)\|$")
plt.title("Error numérico de periodicidad de P(t)")
plt.grid(True)
plt.show()

print("Máximo error de periodicidad:", np.max(periodicity_errors))
print("Promedio del error de periodicidad:", np.mean(periodicity_errors))

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.semilogy(t, reconstruction_error + 1e-18)
plt.xlabel("t")
plt.ylabel(r"$\|\Phi(t)-P(t)e^{Rt}\|$")
plt.title("Error de reconstrucción de la matriz fundamental")
plt.grid(True)
plt.show()

print("Máximo error de reconstrucción:", np.max(reconstruction_error))

## 6. Intensidad de la modulación intraperiódica

Una forma simple de cuantificar qué tan diferente es $P(t)$ de la identidad es calcular

$$
\eta(t)=\|P(t)-I\|.
$$

Si $\eta(t)$ es pequeña, la dinámica efectiva $e^{Rt}$ se parece mucho a la trayectoria completa. Si $\eta(t)$ es grande, la mirada estroboscópica captura correctamente la huella de cada ciclo, pero no describe bien lo que ocurre dentro del ciclo.

In [ ]:
I = np.eye(2)
eta = np.array([np.linalg.norm(P - I) for P in P_t])

plt.figure(figsize=(9, 3.5))
plt.plot(t, eta)
for k in range(n_periods + 1):
    plt.axvline(k*T, linestyle=":", linewidth=1)
plt.xlabel("t")
plt.ylabel(r"$\eta(t)=\|P(t)-I\|$")
plt.title("Medida de modulación intraperiódica")
plt.grid(True)
plt.show()

print("eta mínima:", np.min(eta))
print("eta máxima:", np.max(eta))

## 7. Comparación entre trayectoria real y trayectoria efectiva

Para una condición inicial $x_0$, la trayectoria real es

$$
x(t)=\Phi(t)x_0,
$$

mientras que la trayectoria efectiva LTI es

$$
z(t)=e^{Rt}x_0.
$$

La descomposición de Floquet dice que

$$
x(t)=P(t)z(t).
$$

Por eso, la diferencia entre $x(t)$ y $z(t)$ dentro de un período está gobernada por $P(t)-I$.

In [ ]:
x0 = np.array([1.0, 0.0])

x_t = np.einsum("tij,j->ti", Phi_t, x0)
z_t = np.array([expm(R*tt) @ x0 for tt in t])
z_t = np.real_if_close(z_t, tol=1000)

plt.figure(figsize=(10, 4))
plt.plot(t, x_t[:, 0], label=r"$q(t)$ sistema periódico")
plt.plot(t, z_t[:, 0], "--", label=r"$q_{eff}(t)$ sistema LTI efectivo")
for k in range(n_periods + 1):
    plt.axvline(k*T, linestyle=":", linewidth=1)
plt.xlabel("t")
plt.ylabel("primera variable de estado")
plt.title("Trayectoria real vs. trayectoria efectiva")
plt.legend()
plt.grid(True)
plt.show()

# Norma de la diferencia entre ambas trayectorias
trajectory_difference = np.linalg.norm(x_t - z_t, axis=1)

plt.figure(figsize=(9, 3.5))
plt.plot(t, trajectory_difference)
for k in range(n_periods + 1):
    plt.axvline(k*T, linestyle=":", linewidth=1)
plt.xlabel("t")
plt.ylabel(r"$\|x(t)-z(t)\|$")
plt.title("Diferencia intraperiódica entre trayectoria real y efectiva")
plt.grid(True)
plt.show()

## 8. Verificación estroboscópica

Aunque $x(t)$ y $z(t)$ pueden diferir dentro del período, deben coincidir en los instantes $t=kT$ si se usa la misma matriz efectiva $R$ construida a partir de $M$:

$$
x(kT)=z(kT)=M^kx_0.
$$

In [ ]:
print("Comparación estroboscópica:")
for k in range(n_periods + 1):
    tk = k*T
    xk = np.linalg.matrix_power(M, k) @ x0
    zk = expm(R*tk) @ x0
    diff = np.linalg.norm(xk - zk)
    print(f"k={k}, t={tk:.6f}, ||x(kT)-z(kT)|| = {diff:.3e}")

## 9. Lectura final

Este notebook muestra numéricamente la frase central:

$$
\Phi(t)=P(t)e^{Rt}.
$$

La matriz $R$ captura la evolución efectiva de período en período. La matriz $P(t)$ conserva la modulación intraperiódica.

La reconstrucción

$$
P(t)=\Phi(t)e^{-Rt}
$$

permite observar directamente la parte periódica de la descomposición de Floquet. En este sentido, la teoría no queda como una afirmación abstracta: se convierte en una secuencia verificable de cálculos.